<center><span style="font-family: TimesNewRoman; font-size:1.4em;color:blue;"><b>  LinearRegression:  Predict Price for Used Car in India</b></span></center><br>
<center><img src=https://images.unsplash.com/photo-1506883968894-6e7738ccfc05?ixid=MnwxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8&ixlib=rb-1.2.1&auto=format&fit=crop&w=800&q=80 width="400" height="300"></center>
<p size=1 ><i>image source:Meriç Dağlı on Unsplash</i></p>
<br>


## Context
    
<p style = "font-size : 15px ; color: black;font-family:TimesNewRoman">
There is a huge demand for used cars in the Indian Market today. As sales of new cars have slowed down in the recent past, the pre-owned car market has continued to grow over the past years and is larger than the new car market now. Cars4U is a budding tech start-up that aims to find footholes in this market.
In 2018-19, while new car sales were recorded at 3.6 million units, around 4 million second-hand cars were bought and sold. There is a slowdown in new car sales and that could mean that the demand is shifting towards the pre-owned market. In fact, some car sellers replace their old cars with pre-owned cars instead of buying new ones. Unlike new cars, where price and supply are fairly deterministic and managed by OEMs (Original Equipment Manufacturer / except for dealership level discounts which come into play only in the last stage of the customer journey), used cars are very different beasts with huge uncertainty in both pricing and supply. Keeping this in mind, the pricing scheme of these used cars becomes important in order to grow in the market. We have to come up with a pricing model that can effectively predict the price of used cars and can help the business in devising profitable strategies using differential pricing.</p>


## Data Set
<br>
<p style = "font-size : 15px ; color: black;font-family:TimesNewRoman">
    
1. S.No. : Serial Number<br>
    
2. Name : Name of the car which includes Brand name and Model name<br>
    
3. Location : The location in which the car is being sold or is available for purchase Cities<b<br>r>
    
4. Year : Manufacturing year of the car<br>
    
5. Kilometers_driven : The total kilometers driven in the car by the previous owner(s) in KM.<br>
    
6. Fuel_Type : The type of fuel used by the car. (Petrol, Diesel, Electric, CNG, LPG)<br>
    
7. Transmission : The type of transmission used by the car. (Automatic / Manual)<br>
    
8. Owner : Type of ownership<br>
    
9. Mileage : The standard mileage offered by the car company in kmpl or km/kg<br>
    
10. Engine : The displacement volume of the engine in CC.<br>
    
11. Power : The maximum power of the engine in bhp.<br>
    
12. Seats : The number of seats in the car.<br>
    
13. New_Price : The price of a new car of the same model in INR Lakhs.(1 Lakh = 100, 000)<br>
    
14. Price : The price of the used car in INR Lakhs (1 Lakh = 100, 000)<br>
</p>


## Problem
<p style = "font-size : 15px ; color: black;font-family:TimesNewRoman">

- Does various predicating factors effect the price of the used car .?<br>
- What all  independent variables effect the pricing of used cars?<br>
- Does name of a car have any effect on  pricing of car.?<br>
- How does type of Transmission  effect  pricing?<br>
- Does Location in which the car being sold has any effect on the price?<br>
- Does kilometers_Driven,Year of manufacturing  have negative correlation with  price of the car?<br>
- Does Mileage ,Engine and Power have any effect on the pricing of the car?<br>
- How does number of seat ,Fuel type effect the pricing.?<br>
</p>


# Libraries


In [ ]:
### IMPORT: ------------------------------------
import scipy.stats as stats 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore') # To supress warnings
 # set the background for the graphs
from scipy.stats import skew
plt.style.use('ggplot')
import missingno as msno # to get visualization on missing values
from sklearn.model_selection import train_test_split # Sklearn package's randomized data splitting function
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import math
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth',400)
pd.set_option('display.float_format', lambda x: '%.5f' % x) # To supress numerical display in scientific notations
import statsmodels.api as sm
print("Load Libraries- Done")


# Read and Understand data


In [ ]:
#Reading the csv file  used car data.csv 
data_path = r"C:\Users\suraj\Desktop\used_car_price_prediction\used_cars_data.csv"

df = pd.read_csv(data_path, index_col=0)
cars = df.copy()

print(f"There are {cars.shape[0]} rows and {cars.shape[1]} columns.")


In [ ]:
# inspect data, print top 5 
cars.head(5)


In [ ]:
# bottom 5 rows:
cars.tail(5)


In [ ]:
#get the size of dataframe
print ("Rows     : " , cars.shape[0])  #get number of rows/observations
print ("Columns  : " , cars.shape[1]) #get number of columns
print ("#"*40,"\n","Features : \n\n", cars.columns.tolist()) #get name of columns/features
print ("#"*40,"\nMissing values :\n\n", cars.isnull().sum().sort_values(ascending=False))
print( "#"*40,"\nPercent of missing :\n\n", round(cars.isna().sum() / cars.isna().count() * 100, 2)) # looking at columns with most Missing Values
print ("#"*40,"\nUnique values :  \n\n", cars.nunique())  #  count of unique values


In [ ]:
cars.info()


In [ ]:
#Visualize missing values
msno.bar(cars)


<p style = "font-size : 20px ; color: blue;font-family:TimesNewRoman">
<b>Observations</b></p>
This preview  shows that some columns potentially have a lot of missingness so we'll want to make sure to look into that later.

-  **`New_Price`** has only 1006 values. 86 % values are missing

-  **`Price`**, which is a Target variable 17 % missing values.This needs to be analysed further.

-  **`Seats`** has only 53 values missing and number of seats can be one of key factor in deciding price.
-  **`Power`** and **`Engine`** has 46 missing values.

-  **`Mileage`** only has two values missing.

-  **`Mileage`,`Power`,`Engine`,`New_Price`** we know are quantitative variables but are of object dtype here and needs to to converted to numeric.


In [ ]:
# Making a list of all categorical variables
cat_col = [
    "Fuel_Type",
    "Location",
    "Transmission",
    "Seats",
    "Year",
    "Owner_Type",
    
]
# Printing number of count of each unique value in each column
for column in cat_col:
    print(cars[column].value_counts())
    print("#" * 40)


<p style = "font-size : 20px ; color: blue;font-family:TimesNewRoman">
    <b>  Observations</b></p>

 - Maximum car being sold have fuel type as Diesel.
 - Mumbai has highest numbers of car availabe for purchase.
 - 5204 cars with Manual transmission are available for purchase.
 - Most of the cars are 5 seaters and First owned.
 - Years of car ranges form 1996- 2015


# Data Preprocessing


### Processing Engine,Power ,Mileage columns


Datatype for Engine ,Power and Mileage  are object because of unit assigned ,so striping  units.


In [ ]:
#np.random.seed(9)
cars[['Engine','Power','Mileage']].sample(10)


In [ ]:
typeoffuel=['CNG','LPG']
cars.loc[cars.Fuel_Type.isin(typeoffuel)].head(10)


Power has some values as "nullbhp" .Mileage also has some observations as 0. For fuel type and CNG and LPG mileage is measured in km/kg where as for other type it is measured in kmpl. Since  those units are in  km for both of them no need of conversion . Dropping units from mileages,Engine and Power.


### Mileage


In [ ]:
cars["Mileage"] = cars["Mileage"].str.rstrip(" kmpl")
cars["Mileage"] = cars["Mileage"].str.rstrip(" km/g")


### Engine 


In [ ]:
#remove units
cars["Engine"] = cars["Engine"].str.rstrip(" CC")


### Power


In [ ]:
#remove bhp and replace null with nan
cars["Power"] = cars["Power"].str.rstrip(" bhp")
cars["Power"]= cars["Power"].replace(regex="null", value = np.nan)


In [ ]:
#verify the data
num=['Engine','Power','Mileage']
cars[num].sample(20)


I had seen some values in Power and Mileage as 0.0 so verifying data for Engine, Power, Mileage. Will check once again after converting datatype


In [ ]:
cars.query("Power == '0.0'")['Power'].count()


In [ ]:
cars.query("Mileage == '0.0'")['Mileage'].count()


Converting this observations to Nan so we will remember to handle them when handling missing values.


In [ ]:
cars.loc[cars["Mileage"]=='0.0','Mileage']=np.nan


In [ ]:
cars.loc[cars["Engine"]=='0.0','Engine'].count()


In [ ]:
cars[num].nunique()


In [ ]:
cars[num].isnull().sum()


There are 46 missing values in Engine, 175 in  Power,83 in Mileage. 


###  Processing Seats


In [ ]:
cars.query("Seats == 0.0")['Seats']


In [ ]:
#seats cannot be 0 so changing it to nan and will be handled in missing value
cars.loc[3999,'Seats'] =np.nan


###  Processing  New Price
We know that `New_Price` is the price of a new car of the same model in INR Lakhs.(1 Lakh = 100, 000)

This column clearly has a lot of missing values. We will impute the missing values later. For now we will only extract the numeric values from this column.


In [ ]:
# Create a new column after splitting the New_Price values.
import re

new_price_num = []

# Regex for numeric + " " + "Lakh"  format
regex_power = "^\d+(\.\d+)? Lakh$"

for observation in df["New_Price"]:
    if isinstance(observation, str):
        if re.match(regex_power, observation):
            new_price_num.append(float(observation.split(" ")[0]))
        else:
            # To detect if there are any observations in the column that do not follow [numeric + " " + "Lakh"]  format
            # that we see in the sample output
            print(
                "The data needs furthur processing.mismatch ",
                observation,
            )
    else:
        # If there are any missing values in the New_Price column, we add missing values to the new column
        new_price_num.append(np.nan)


In [ ]:
new_price_num = []

for observation in df["New_Price"]:
    if isinstance(observation, str):
        if re.match(regex_power, observation):
            new_price_num.append(float(observation.split(" ")[0]))
        else:
            # Converting values in Crore to lakhs
            new_price_num.append(float(observation.split(" ")[0]) * 100)
    else:
        # If there are any missing values in the New_Price column, we add missing values to the new column
        new_price_num.append(np.nan)

# Add the new column to the data
cars["new_price_num"] = new_price_num

# Checking the new dataframe
cars.head(5)  # Looks ok


# Feature Enginering


## converting datatype


In [ ]:
#converting object data type to category data type
cars["Fuel_Type"] = cars["Fuel_Type"].astype("category")
cars["Transmission"] = cars["Transmission"].astype("category")
cars["Owner_Type"] = cars["Owner_Type"].astype("category")
#converting datatype  
cars["Mileage"] = cars["Mileage"].astype(float)
cars["Power"] = cars["Power"].astype(float)
cars["Engine"]=cars["Engine"].astype(float)


In [ ]:
cars.describe().T


### Processing Years to Derive Age of car
Since year has 2014, 1996  etc. But this will not help to understand how old cars is and its effect on  price.
so creating  two new columns data collection year and Age . Data collection year would be 2020 and Age column would be Ageofcar = 2020 - year. And then drop currentyear columns


In [ ]:
cars['Current_year']=2026
cars['Ageofcar']=cars['Current_year']-cars['Year']
cars.drop('Current_year',axis=1,inplace=True)
cars.head()


### Processing Name column


Brands do play an important role in Car selection and Prices. So extracting brand names from the Name.


In [ ]:
#dropping rows with name as null
cars = cars.dropna(subset=['Name'])


In [ ]:
#As mentioned in dataset car name has Brand and model so extracting it ,This can help to fill missing values of price column as brand 
cars['Brand'] = cars['Name'].str.split(' ').str[0] #Separating Brand name from the Name
cars['Model'] = cars['Name'].str.split(' ').str[1] + cars['Name'].str.split(' ').str[2]


In [ ]:
cars.Brand.unique()


In [ ]:
col=['ISUZU','Isuzu','Mini','Land']
#correcting brand names
cars[cars.Brand.isin(col)].sample(5)


Brand names like ISUZU and Isuzu are same and needs to be corrected. Land, Mini seems to be incorrect. So correcting brand names.


In [ ]:
cars.info()


In [ ]:
#changing brandnames
cars.loc[cars.Brand == 'ISUZU','Brand']='Isuzu'
cars.loc[cars.Brand=='Mini','Brand']='Mini Cooper'
cars.loc[cars.Brand=='Land','Brand']='Land Rover'
#cars['Brand']=cars["Brand"].astype("category")


In [ ]:
cars.Brand.nunique()


In [ ]:
cars.groupby(cars.Brand).size().sort_values(ascending =False)


There are 32 unique Brands in the dataset.Maruti brand is most available for purchase/Sold followed by Hyundai.


In [ ]:
cars.Model.isnull().sum()


In [ ]:
#drop row with no model
cars.dropna(subset=['Model'],axis=0,inplace=True)


In [ ]:
cars.Model.nunique()


In [ ]:
cars.groupby('Model')['Model'].size().nlargest(30)


There are 726 unique models and Swift Dzire is most popular Model.


In [ ]:
# Save preprocessed data for the next parts
cars.to_csv('used_cars_preprocessed.csv', index=False)
print('Preprocessed data successfully saved to used_cars_preprocessed.csv')
